# KDE-score closure for an entropy-driven inertial flow

This notebook generates `fig:gradflow-second-order-momentum-entropy`.  It illustrates a subtle point: the Shannon entropy
\[
    f(\rho\,dx)=\int \rho(x)\log\rho(x)\,dx
\]
has the Wasserstein velocity \(v_\rho=-\nabla\log\rho\), but it is infinite on empirical Dirac measures.  The particle scheme below is therefore a regularized score closure: a Gaussian KDE estimates \(\rho_t\), and its score is evaluated directly at the current particle positions.  For the Gaussian kernel this score has the closed form
\[
    \nabla\log\widehat\rho_h(x_i)=\frac{\sum_j \exp(-\|x_i-x_j\|^2/(2h^2))(x_j-x_i)}{h^2\sum_j \exp(-\|x_i-x_j\|^2/(2h^2))}.
\]
It is computed by chunked PyTorch tensor operations, rather than by differentiating a density on a grid.  The same force is used either in an overdamped flow or in the Newton lift
\[
    \dot x=s,\qquad \dot s=-\tau\nabla\log\widehat\rho_t(x)-\kappa x.
\]
The term \(-\kappa x\) is a quadratic confining force, included only to keep the entropy-driven expansion in a readable window over a longer simulation time.

The figure uses thousands of particles for the KDE, but displays only a farthest-point subset of trajectories.  Each row shows a trajectory panel on the left and density snapshots across time on the right.


In [1]:
from pathlib import Path
import os
import shutil
import sys

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.collections import LineCollection
from scipy.ndimage import gaussian_filter
from matplotlib.colors import to_rgb

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "notebooks-figures"))

from figure_style import BLUE, RED, figure_dir, interp_color, remove_axes, save_pdf, setup_matplotlib

setup_matplotlib()
torch.set_num_threads(min(6, max(1, torch.get_num_threads())))
rng = np.random.default_rng(20260621)
out_dir = figure_dir("gradflow-second-order-momentum-entropy")
thumb_dir = ROOT / "notebooks-figures" / "thumbnails"
thumb_dir.mkdir(parents=True, exist_ok=True)


## Particle cloud and direct KDE score

The initial law is a compact three-component Gaussian mixture whose components have visibly different covariance matrices.  At each time step, the force is computed from the exact Gaussian KDE score of the current particle cloud, using a chunked PyTorch implementation.  The bandwidth is intentionally smaller than in the grid-based draft so that the particles react to more local density variations.  The finite grid below is used only later to render smooth density images.


In [2]:
def sample_initial_cloud(n=5000):
    weights = np.array([0.36, 0.34, 0.30])
    means = np.array([[-0.72, 0.30], [0.55, 0.12], [-0.05, -0.58]])
    covs = np.array([
        [[0.020, 0.004], [0.004, 0.040]],
        [[0.050, -0.010], [-0.010, 0.024]],
        [[0.030, 0.000], [0.000, 0.060]],
    ])
    labels = rng.choice(len(weights), size=n, p=weights)
    x = np.empty((n, 2), dtype=np.float32)
    for k in range(len(weights)):
        mask = labels == k
        x[mask] = rng.multivariate_normal(means[k], covs[k], size=int(mask.sum()))
    x -= x.mean(axis=0, keepdims=True)
    return x

X0 = sample_initial_cloud()
N = len(X0)

bandwidth = 0.16
score_clip = 2.20
entropy_strength = 0.17
confinement_strength = 0.35
total_time = 4.20


def kde_score_torch(points, chunk_size=512):
    """Evaluate the Gaussian KDE score directly at the particles.

    The centers are the current particles, treated as fixed KDE centers while
    differentiating with respect to the query location. This is the analytic
    PyTorch implementation of grad log rho_hat_h, equivalent to autograd on
    the Gaussian KDE but cheaper and less memory hungry.
    """
    X = torch.as_tensor(points, dtype=torch.float32)
    h2 = float(bandwidth * bandwidth)
    scores = []
    with torch.no_grad():
        for start in range(0, len(points), chunk_size):
            query = X[start : start + chunk_size]
            diff = X.unsqueeze(0) - query.unsqueeze(1)
            weights = torch.exp(-0.5 * torch.sum(diff * diff, dim=2) / h2)
            density = torch.sum(weights, dim=1).clamp_min(1e-12)
            numerator = torch.einsum("bn,bnd->bd", weights, diff)
            scores.append(numerator / (h2 * density[:, None]))
    score = torch.cat(scores, dim=0).cpu().numpy()
    norms = np.linalg.norm(score, axis=1)
    score *= np.minimum(1.0, score_clip / (norms + 1e-12))[:, None]
    return score


## Time integration

We keep only the two regimes used in the paper figure: the overdamped score flow and the undamped Newton lift.  Both start from the same particles and, for Newton, zero initial velocity.  The simulation is run long enough for the quadratic confinement to be visible.


In [3]:
def entropy_confinement_force(X):
    score = kde_score_torch(X)
    return -entropy_strength * score - confinement_strength * X


def simulate_first_order(dt=0.012, record_every=4):
    X = X0.copy()
    n_steps = int(round(total_time / dt))
    records = []
    for k in range(n_steps + 1):
        if k % record_every == 0 or k == n_steps:
            records.append(X.copy())
        if k < n_steps:
            X = X + dt * entropy_confinement_force(X)
    return np.asarray(records)


def simulate_newton(dt=0.008, record_every=7):
    X = X0.copy()
    S = np.zeros_like(X)
    n_steps = int(round(total_time / dt))
    records = []
    for k in range(n_steps + 1):
        if k % record_every == 0 or k == n_steps:
            records.append(X.copy())
        if k < n_steps:
            S = S + dt * entropy_confinement_force(X)
            X = X + dt * S
    return np.asarray(records)

records = {
    "no_momentum": simulate_first_order(),
    "newton": simulate_newton(),
}

# Representative trajectories: avoid extreme tails, then farthest-point sample.
center0 = X0.mean(axis=0)
C0_inv = np.linalg.inv(np.cov(X0.T))
mahal = np.einsum("ni,ij,nj->n", X0 - center0, C0_inv, X0 - center0)
candidates = np.flatnonzero(mahal <= np.quantile(mahal, 0.94))
P = X0[candidates]
chosen = [int(np.argmin(P[:, 0]))]
dist2 = np.sum((P - P[chosen[0]]) ** 2, axis=1)
for _ in range(1, 58):
    j = int(np.argmax(dist2))
    chosen.append(j)
    dist2 = np.minimum(dist2, np.sum((P - P[j]) ** 2, axis=1))
traj_ids = candidates[np.asarray(chosen)]

{k: v.shape for k, v in records.items()}


{'no_momentum': (89, 5000, 2), 'newton': (76, 5000, 2)}

## Rendering helpers

The trajectory panel overlays paths with a few density contours.  The density snapshots are rendered as colored images with the same spatial window in every panel, so the reader can compare the overdamped and Newton rows directly.  These rasterized densities are only for display; they are not used in the particle dynamics.


In [4]:
all_points = np.concatenate([r.reshape(-1, 2) for r in records.values()], axis=0)
lo = all_points.min(axis=0)
hi = all_points.max(axis=0)
center = 0.5 * (lo + hi)
span = max(hi - lo)
pad = 0.06 * span
xlim_plot = (center[0] - 0.5 * span - pad, center[0] + 0.5 * span + pad)
ylim_plot = (center[1] - 0.5 * span - pad, center[1] + 0.5 * span + pad)

plot_n = 250
plot_x = np.linspace(*xlim_plot, plot_n)
plot_y = np.linspace(*ylim_plot, plot_n)
PX, PY = np.meshgrid(plot_x, plot_y)

def display_density(points, sigma=5.0):
    H, _, _ = np.histogram2d(points[:, 0], points[:, 1], bins=plot_n, range=[xlim_plot, ylim_plot], density=False)
    D = gaussian_filter(H.T, sigma=sigma, mode="nearest")
    return D / (D.max() + 1e-12)


def density_rgb(D, color, gamma=0.76):
    q = np.quantile(D[D > 0], 0.985) if np.any(D > 0) else 1.0
    A = np.clip(D / (q + 1e-12), 0, 1) ** gamma
    base = np.array(to_rgb(color))
    return 1.0 - A[..., None] * (1.0 - base)


def contour_levels(D):
    return [0.24, 0.48, 0.72]


def add_colored_trajectory(ax, path, lw=0.42, alpha=0.48):
    segments = np.stack([path[:-1], path[1:]], axis=1)
    colors = [(*interp_color(t), alpha) for t in np.linspace(0, 1, len(segments))]
    ax.add_collection(LineCollection(segments, colors=colors, linewidths=lw, zorder=4))

snapshot_fracs = [0.0, 0.25, 0.50, 0.75, 1.0]
snapshot_labels = ["000", "025", "050", "075", "100"]
snapshot_colors = [interp_color(t) for t in snapshot_fracs]


def snapshot_ids(rec):
    return [int(frac * (len(rec) - 1)) for frac in snapshot_fracs]


def render_trajectory_panel(rec, filename, *, fig_size=(1.28, 1.28)):
    fig, ax = plt.subplots(figsize=fig_size)
    ax.set_xlim(*xlim_plot)
    ax.set_ylim(*ylim_plot)
    ax.set_aspect("equal")
    ax.set_facecolor("white")
    for sid, color, alpha in zip(snapshot_ids(rec), snapshot_colors, [0.36, 0.40, 0.45, 0.50, 0.62]):
        D = display_density(rec[sid])
        ax.contour(PX, PY, D, levels=contour_levels(D), colors=[color], linewidths=0.44, alpha=alpha, zorder=2)
    for idx in traj_ids:
        add_colored_trajectory(ax, rec[:, idx, :])
    ax.scatter(rec[0, traj_ids, 0], rec[0, traj_ids, 1], s=2.7, color=RED, edgecolor="none", alpha=0.74, zorder=5)
    ax.scatter(rec[-1, traj_ids, 0], rec[-1, traj_ids, 1], s=2.7, color=BLUE, edgecolor="none", alpha=0.82, zorder=5)
    remove_axes(ax)
    fig.subplots_adjust(0, 0, 1, 1)
    save_pdf(fig, out_dir / filename, pad_inches=0.004)
    return fig


def render_density_panel(points, color, filename, *, fig_size=(1.28, 1.28)):
    D = display_density(points)
    fig, ax = plt.subplots(figsize=fig_size)
    ax.imshow(
        density_rgb(D, color),
        extent=[*xlim_plot, *ylim_plot],
        origin="lower",
        interpolation="bilinear",
        zorder=1,
    )
    ax.contour(PX, PY, D, levels=contour_levels(D), colors=[color], linewidths=0.40, alpha=0.42, zorder=2)
    ax.set_xlim(*xlim_plot)
    ax.set_ylim(*ylim_plot)
    ax.set_aspect("equal")
    ax.set_facecolor("white")
    remove_axes(ax)
    fig.subplots_adjust(0, 0, 1, 1)
    save_pdf(fig, out_dir / filename, pad_inches=0.004)
    return fig

output_specs = {
    "no_momentum": "no-momentum",
    "newton": "newton",
}

for key, prefix in output_specs.items():
    rec = records[key]
    fig = render_trajectory_panel(rec, f"{prefix}-trajectories.pdf")
    plt.close(fig)
    for lab, sid, color in zip(snapshot_labels, snapshot_ids(rec), snapshot_colors):
        fig = render_density_panel(rec[sid], color, f"{prefix}-density-t{lab}.pdf")
        plt.close(fig)

# Mirror the paper panels for the independent PDE4ML survey.
pde_dir = ROOT / "PDE4ML" / "figures" / out_dir.name
pde_dir.mkdir(parents=True, exist_ok=True)
for pdf in out_dir.glob("*.pdf"):
    shutil.copy2(pdf, pde_dir / pdf.name)


## Contact-sheet thumbnail

The paper assembles the PDF panels directly in LaTeX.  This thumbnail is only used by the figure-notebook gallery.

In [5]:
labels = [("GF", "no_momentum"), ("Newton", "newton")]
fig, axes = plt.subplots(2, 6, figsize=(7.7, 2.6), constrained_layout=False)
for row, (label, key) in enumerate(labels):
    rec = records[key]
    ax = axes[row, 0]
    ax.set_xlim(*xlim_plot); ax.set_ylim(*ylim_plot); ax.set_aspect("equal"); ax.set_facecolor("white")
    for sid, color in zip(snapshot_ids(rec), snapshot_colors):
        D = display_density(rec[sid])
        ax.contour(PX, PY, D, levels=contour_levels(D), colors=[color], linewidths=0.38, alpha=0.48, zorder=2)
    for idx in traj_ids:
        add_colored_trajectory(ax, rec[:, idx, :], lw=0.30, alpha=0.36)
    ax.text(0.04, 0.94, label, transform=ax.transAxes, ha="left", va="top", fontsize=7.5, color="#262626")
    remove_axes(ax)
    for col, (sid, color) in enumerate(zip(snapshot_ids(rec), snapshot_colors), start=1):
        ax = axes[row, col]
        D = display_density(rec[sid])
        ax.imshow(density_rgb(D, color), extent=[*xlim_plot, *ylim_plot], origin="lower", interpolation="bilinear")
        ax.contour(PX, PY, D, levels=contour_levels(D), colors=[color], linewidths=0.34, alpha=0.40)
        ax.set_xlim(*xlim_plot); ax.set_ylim(*ylim_plot); ax.set_aspect("equal"); ax.set_facecolor("white")
        remove_axes(ax)
fig.subplots_adjust(left=0.004, right=0.996, top=0.996, bottom=0.004, wspace=0.025, hspace=0.030)
thumb_path = thumb_dir / "gradflow-second-order-momentum-entropy.png"
fig.savefig(thumb_path, dpi=205, bbox_inches="tight", pad_inches=0.012)
plt.close(fig)
thumb_path


PosixPath('/Users/gpeyre/Dropbox/github/ot4ml/notebooks-figures/thumbnails/gradflow-second-order-momentum-entropy.png')